# Male-specific contigs: which chromosome is the Y?

Contigs assembled from **male-specific 31-mers** were BLASTed against a **single-haplotype**
assembly with 13 chromosomes. This notebook filters the hits, classifies each contig by where
it lands, and counts the contigs per chromosome. No chromosome is labelled in advance: the one
that collects the most **single-hit** contigs, in total and per Mb, is the Y candidate.

## 1. Settings

In [ ]:
BLAST_TSV = "malehits.tsv"   # BLAST tabular output
# the fields of the -outfmt 6 string used for BLAST, in order
OUTFMT    = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore"
N_CHROM   = 13               # the N_CHROM longest scaffolds are the chromosomes

# a hit must pass all four filters
MIN_PIDENT = 95.0            # percent identity
MIN_QCOV   = 0.80            # aligned length / contig length
MAX_EVALUE = 1e-10
MIN_QLEN   = 50              # contig length (bp)

# genome-wide density plot (section 7)
WINDOW    = 10_000           # window size (bp)
ROLLING_K = 50               # rolling mean width (windows)

## 2. Load and filter the hits

Each row is one HSP. The chromosomes are the `N_CHROM` longest scaffolds in the file (printed
below, worth a check); any other scaffold is treated as unplaced.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'single hit': '#2a78d6', 'repeated': '#eb6834', 'multi': '#1baf7a'}
CLASSES = list(COLORS)

raw = pd.read_csv(BLAST_TSV, sep='\t', names=OUTFMT.split(), comment='#',
                  dtype={'qseqid': str, 'sseqid': str})
raw['qcov'] = raw['length'] / raw['qlen']
hits = raw[(raw['qlen'] >= MIN_QLEN) & (raw['pident'] >= MIN_PIDENT) &
           (raw['qcov'] >= MIN_QCOV) & (raw['evalue'] <= MAX_EVALUE)].copy()
hits['pos'] = (hits['sstart'] + hits['send']) / 2          # hit midpoint on the scaffold

chrom_len = raw.groupby('sseqid')['slen'].max().nlargest(N_CHROM)   # longest first
chroms = chrom_len.index

print(f"{len(raw):,} HSPs from {raw['qseqid'].nunique():,} contigs; "
      f"{len(hits):,} HSPs from {hits['qseqid'].nunique():,} contigs pass the filters")
print('Chromosomes:', ', '.join(chroms))

## 3. Classify each contig

| class | the contig's passing hits |
|---|---|
| **single hit** | one HSP on one scaffold: the cleanest placement |
| **repeated** | several HSPs, all on the same scaffold |
| **multi** | two or more scaffolds: not uniquely placeable |

Saved to `contig_classification.tsv`, together with the best-scoring scaffold of each contig
and every scaffold it hits.

In [ ]:
g = hits.groupby('qseqid')
best = hits.loc[g['bitscore'].idxmax()].set_index('qseqid')    # best-scoring HSP per contig

per_contig = pd.DataFrame({
    'n_scaffolds':   g['sseqid'].nunique(),
    'n_hsp':         g.size(),
    'best_scaffold': best['sseqid'],
    'best_bitscore': best['bitscore'],
    'all_scaffolds': g['sseqid'].agg(lambda s: ','.join(sorted(set(s)))),
})
per_contig['classification'] = np.select(
    [per_contig['n_scaffolds'] > 1, per_contig['n_hsp'] > 1], ['multi', 'repeated'], 'single hit')

per_contig.to_csv('contig_classification.tsv', sep='\t', index_label='contig')
per_contig['classification'].value_counts()

## 4. Per-chromosome summary

Contigs per chromosome by class (a `multi` contig counts on every chromosome it hits), and
single hits per Mb, because a longer chromosome collects more hits by chance alone. Sorted by
single hits, so the Y candidate should top the table. Saved to `chromosome_summary.tsv`.

In [ ]:
# one row per (contig, scaffold it hits)
touch = (per_contig.assign(scaffold=per_contig['all_scaffolds'].str.split(','))
                   .explode('scaffold', ignore_index=True))
summary = (pd.crosstab(touch['scaffold'], touch['classification'])
             .reindex(index=chroms, columns=CLASSES, fill_value=0)
             .rename_axis(index='chromosome', columns=None))
summary.insert(0, 'length_mb', chrom_len / 1e6)
summary['single_hit_per_mb'] = summary['single hit'] / summary['length_mb']
summary = summary.round(3).sort_values('single hit', ascending=False)
summary.to_csv('chromosome_summary.tsv', sep='\t')

unplaced = (per_contig['classification'] == 'single hit') & ~per_contig['best_scaffold'].isin(chroms)
print(f'{unplaced.sum():,} single-hit contigs fall on unplaced scaffolds and are not counted below')
summary

## 5. Where the single-hit contigs fall

One row per chromosome, longest on top, drawn to scale; one tick per single-hit contig, so
darker stretches hold more hits.

In [ ]:
single = per_contig.index[per_contig['classification'] == 'single hit']
sh = hits[hits['qseqid'].isin(single) & hits['sseqid'].isin(chroms)]    # one HSP per contig
row = {c: i for i, c in enumerate(chroms)}

fig, ax = plt.subplots(figsize=(12, 0.4 * len(chroms) + 1.5))
ax.hlines(range(len(chroms)), 0, chrom_len / 1e6, color='#e1e0d9', lw=4, zorder=1)
ax.scatter(sh['pos'] / 1e6, sh['sseqid'].map(row), marker='|', s=150, alpha=0.3,
           color=COLORS['single hit'], linewidths=1, zorder=2)
ax.set_yticks(range(len(chroms)), [f"{c}  (n={summary.at[c, 'single hit']:,})" for c in chroms])
ax.invert_yaxis()
ax.set_xlim(left=0)
ax.set_xlabel('Position (Mb)')
ax.set_title(f'Single-hit contig positions ({len(sh):,} contigs)')
fig.savefig('single_hit_placements.png')
plt.show()

## 6. Counts per chromosome

Left: contigs per chromosome by class. Right: single hits per Mb. Same row order as above.

In [ ]:
s = summary.loc[chroms]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 0.4 * len(chroms) + 1.5), sharey=True)
s[CLASSES].plot.barh(stacked=True, ax=ax1, width=0.7, color=COLORS, edgecolor='white')
s['single_hit_per_mb'].plot.barh(ax=ax2, width=0.7, color=COLORS['single hit'])
ax1.invert_yaxis()
ax1.set(xlabel='contigs', ylabel='', title='Contigs per chromosome')
ax2.set(xlabel='single-hit contigs per Mb', title='Single hits per Mb')
ax1.legend(frameon=False)
fig.savefig('chromosome_counts.png')
plt.show()

## 7. Single hits along the genome

The chromosomes laid end to end, longest first. Top: single-hit contigs in each `WINDOW`-bp
window (empty windows not drawn). Bottom: their rolling mean over `ROLLING_K` windows, which
shows where on a chromosome the hits concentrate.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
offset = 0
for i, c in enumerate(chroms):
    counts = np.bincount((sh.loc[sh['sseqid'] == c, 'pos'] // WINDOW).astype(int),
                         minlength=int(chrom_len[c] // WINDOW) + 1)
    x = offset + np.arange(len(counts)) * WINDOW
    color = ('#898781', COLORS['single hit'])[i % 2]          # alternate per chromosome
    ax1.scatter(x[counts > 0], counts[counts > 0], s=6, color=color, linewidths=0)
    ax2.plot(x, pd.Series(counts).rolling(ROLLING_K, center=True, min_periods=1).mean(),
             color=color, lw=1)
    offset += chrom_len[c]

ax2.set_xticks(chrom_len.cumsum() - chrom_len / 2, chroms, rotation=45, ha='right')
ax2.set_xlim(0, offset)
ax1.set_ylim(bottom=0)
ax1.set(ylabel=f'single hits / {WINDOW // 1000} kb', title='Single-hit contigs along the genome')
ax2.set(ylabel=f'rolling mean ({ROLLING_K} windows)')
fig.savefig('single_hit_manhattan.png')
plt.show()